# 🌤️ Weather Forecasting — Foundation Pipeline

## End-to-End Data Acquisition, Time-Series Structuring & Deep EDA

**Location:** Cairo, Egypt (30.0444°N, 31.2357°E)  
**Period:** January 1, 2023 – December 31, 2025 (3 years, daily)  
**API:** Open-Meteo Historical Weather Archive (free, keyless, ERA5 reanalysis)  
**Variables:** 7 daily features — temperature (min/max/mean), precipitation, humidity, wind speed, cloud cover

---

This notebook walks through the three foundational pillars of the project:

| Part | Topic | Goal |
|------|-------|------|
| **1** | Data Acquisition | Fetch historical weather data from the Open-Meteo API |
| **2** | Dataset Preparation | Parse, validate, clean, and export the time-series |
| **3** | Deep Time-Series EDA | Uncover patterns, seasonality, stationarity, and anomalies |

## Setup & Imports

In [ ]:
import sys
import os
import json
import logging

# Ensure the project root is on the Python path
PROJECT_ROOT = os.path.dirname(os.path.abspath('.'))
if os.path.basename(os.getcwd()) == 'notebooks':
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Project modules
import config
from src import data_acquisition, data_preparation, eda

# Plotting setup
%matplotlib inline
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 150, 'figure.figsize': (14, 5)})

# Logging
logging.basicConfig(level=logging.INFO, format='%(levelname)-7s | %(name)s | %(message)s')

print(f'Project root: {PROJECT_ROOT}')
print(f'Location: {config.LOCATION_NAME}')
print(f'Period: {config.START_DATE} → {config.END_DATE}')

---

# Part 1: Weather API Integration & Data Acquisition

We use the **Open-Meteo Historical Weather API** to fetch 3 years of daily weather records for Cairo.

### Why Open-Meteo?
| Criteria | Open-Meteo | OpenWeatherMap | NASA POWER |
|----------|-----------|----------------|------------|
| **Auth** | No API key | API key required | No key |
| **Cost** | Free (non-commercial) | Limited free tier | Free |
| **Historical depth** | Back to 1940 (ERA5) | 5 years max (free) | Back to 1981 |
| **Rate limit** | 10,000/day | 60/min (free) | 100/day |
| **Speed** | Fast (single call) | Paginated | Bulk download |

In [ ]:
# Fetch weather data from the API
raw_data = data_acquisition.run()

In [ ]:
# Inspect the raw response structure
print('Top-level keys:', list(raw_data.keys()))
print(f'\nLocation metadata:')
for key in ['latitude', 'longitude', 'timezone', 'elevation', 'generationtime_ms']:
    if key in raw_data:
        print(f'  {key}: {raw_data[key]}')

print(f'\nDaily variables: {list(raw_data["daily"].keys())}')
print(f'Number of records: {len(raw_data["daily"]["time"])}')
print(f'Date range: {raw_data["daily"]["time"][0]} → {raw_data["daily"]["time"][-1]}')

In [ ]:
# Preview raw data (first 5 days)
preview = {k: v[:5] for k, v in raw_data['daily'].items()}
pd.DataFrame(preview)

The raw JSON has been saved to `data/raw/open_meteo_response.json` for full reproducibility.

**API documentation:**
- **Endpoint:** `https://archive-api.open-meteo.com/v1/archive`
- **Parameters:** latitude, longitude, start_date, end_date, daily variables, timezone
- **Coordinates:** 30.0444°N, 31.2357°E (Cairo, Egypt)
- **Rate limit:** 10,000 requests/day — we used 1 request
- **Data source:** ECMWF ERA5 reanalysis

---

# Part 2: Time-Series Dataset Preparation

We now parse the raw JSON into a clean pandas DataFrame with:
1. Proper datetime indexing with explicit `freq='D'`
2. Continuity validation (no duplicates, no gaps)
3. Domain-appropriate missing value handling
4. Unit verification and physical bounds checking

In [ ]:
# Run the full data preparation pipeline
df = data_preparation.run(raw_data)

In [ ]:
# Inspect the cleaned DataFrame
print(f'Shape: {df.shape}')
print(f'Index type: {type(df.index).__name__}')
print(f'Frequency: {df.index.freq}')
print(f'Date range: {df.index.min()} → {df.index.max()}')
print(f'NaN count: {df.isna().sum().sum()}')
print()
df.head(10)

In [ ]:
# Data types and memory usage
df.info()

### Missing Value Strategy

| Variable Type | Strategy | Rationale |
|--------------|----------|----------|
| Temperature, Humidity, Wind, Cloud Cover | **Linear interpolation** | Physically smooth, continuous variables — values change gradually day-to-day |
| Precipitation | **Fill with 0.0** | Sparse, event-driven variable (zero-inflated). Missing = no rain event in desert climate. Interpolation would fabricate false rainfall |

### Unit Verification

| Variable | Expected Unit | Verified |
|----------|-------------|----------|
| Temperature (min/max/mean) | °C | ✅ |
| Precipitation | mm | ✅ |
| Humidity | % | ✅ |
| Wind Speed | km/h | ✅ |
| Cloud Cover | % | ✅ |

Clean dataset exported to `data/processed/cairo_weather_clean.csv`.

---

# Part 3: Deep Time-Series Exploratory Data Analysis

## 3.1 Descriptive Statistics & Distributions

In [ ]:
# Summary statistics with skewness and kurtosis
desc = df.describe().T
desc['skew'] = df.skew()
desc['kurtosis'] = df.kurtosis()
desc.round(2)

**Key observations:**
- **Temperature** has a wide range reflecting Cairo's desert climate (cold winter nights to hot summer days)
- **Precipitation** is heavily right-skewed — most days are dry, with rare rainfall events
- **Humidity** shows moderate variability across the year
- **Cloud cover** is generally low, consistent with a desert climate

In [ ]:
# Distribution plots (Histogram + KDE) for all variables
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

colors = sns.color_palette('husl', len(df.columns))
for i, col in enumerate(df.columns):
    ax = axes[i]
    sns.histplot(df[col].dropna(), kde=True, ax=ax, color=colors[i],
                 edgecolor='white', linewidth=0.5, alpha=0.7)
    ax.set_title(f'{col} ({config.UNITS.get(col, "")})', fontweight='bold')
    ax.set_xlabel(config.UNITS.get(col, col))
    ax.set_ylabel('Frequency')

# Hide last unused subplot
axes[-1].set_visible(False)

fig.suptitle('Feature Distributions — Cairo Daily Weather (2023–2025)',
             fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

## 3.2 Temporal Dynamics

### Full-Span Time-Series Plots

In [ ]:
# Temperature time series with min/max range
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 9), sharex=True)

ax1.plot(df.index, df['temp_max_c'], label='Max Temp', color='#e74c3c', alpha=0.8, linewidth=0.8)
ax1.plot(df.index, df['temp_min_c'], label='Min Temp', color='#3498db', alpha=0.8, linewidth=0.8)
ax1.plot(df.index, df['temp_mean_c'], label='Mean Temp', color='#2c3e50', linewidth=1.2)
ax1.fill_between(df.index, df['temp_min_c'], df['temp_max_c'], alpha=0.15, color='#95a5a6')
ax1.set_ylabel('Temperature (°C)')
ax1.set_title('Daily Temperature — Cairo (2023–2025)', fontweight='bold')
ax1.legend(loc='upper right')
ax1.grid(True, alpha=0.3)

# Precipitation
ax2.bar(df.index, df['precipitation_mm'], color='#2980b9', alpha=0.7, width=1.5)
ax2.set_ylabel('Precipitation (mm)')
ax2.set_title('Daily Precipitation — Cairo (2023–2025)', fontweight='bold')
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### Rolling Mean Trend Analysis

Applying 7-day and 30-day rolling windows to smooth out daily noise and reveal the underlying trend.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(df.index, df['temp_mean_c'], alpha=0.3, color='#bdc3c7', linewidth=0.6, label='Daily Mean')
ax.plot(df.index, df['temp_mean_c'].rolling(7).mean(),
        color='#e67e22', linewidth=1.5, label='7-day Rolling Mean')
ax.plot(df.index, df['temp_mean_c'].rolling(30).mean(),
        color='#c0392b', linewidth=2.0, label='30-day Rolling Mean')

ax.set_ylabel('Temperature (°C)')
ax.set_title('Mean Temperature — Rolling Window Trend Analysis', fontweight='bold')
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### Monthly Seasonality Boxplots

Grouping data by month reveals the annual cycle and the variability within each month.

In [ ]:
df_m = df.copy()
df_m['month_name'] = df_m.index.strftime('%b')

month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(x='month_name', y='temp_mean_c', hue='month_name',
            data=df_m, ax=ax1, order=month_order,
            palette='coolwarm', legend=False, fliersize=3)
ax1.set_xlabel('Month')
ax1.set_ylabel('Mean Temperature (°C)')
ax1.set_title('Monthly Temperature Seasonality', fontweight='bold')

sns.boxplot(x='month_name', y='precipitation_mm', hue='month_name',
            data=df_m, ax=ax2, order=month_order,
            palette='Blues', legend=False, fliersize=3)
ax2.set_xlabel('Month')
ax2.set_ylabel('Precipitation (mm)')
ax2.set_title('Monthly Precipitation Seasonality', fontweight='bold')

fig.tight_layout()
plt.show()

### Temperature Volatility

Rolling 30-day standard deviation captures how variable temperatures are across different seasons.

In [ ]:
rolling_std = df['temp_mean_c'].rolling(30).std()

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, rolling_std, color='#8e44ad', linewidth=1.2)
ax.fill_between(df.index, 0, rolling_std, alpha=0.2, color='#8e44ad')
ax.set_ylabel('Rolling Std Dev (°C)')
ax.set_title('Temperature Volatility — 30-day Rolling Std Dev', fontweight='bold')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 3.3 Decomposition & Structural Analysis

### Seasonal Decomposition

Decomposing mean temperature into **Trend**, **Seasonal**, and **Residual** components using additive decomposition with `period=365` (annual cycle).

In [ ]:
series = df['temp_mean_c'].dropna()
decomp = seasonal_decompose(series, model='additive', period=365)

fig, axes = plt.subplots(4, 1, figsize=(16, 12), sharex=True)

axes[0].plot(series.index, series.values, color='#2c3e50', linewidth=0.6)
axes[0].set_ylabel('Observed (°C)')
axes[0].set_title('Seasonal Decomposition of Mean Temperature (Additive, period=365)',
                  fontweight='bold', fontsize=14)

axes[1].plot(decomp.trend.index, decomp.trend.values, color='#e74c3c', linewidth=1.2)
axes[1].set_ylabel('Trend (°C)')

axes[2].plot(decomp.seasonal.index, decomp.seasonal.values, color='#27ae60', linewidth=0.6)
axes[2].set_ylabel('Seasonal (°C)')

axes[3].plot(decomp.resid.index, decomp.resid.values, color='#7f8c8d', linewidth=0.4, alpha=0.8)
axes[3].set_ylabel('Residual (°C)')
axes[3].xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))

for ax in axes:
    ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

### Autocorrelation (ACF) & Partial Autocorrelation (PACF)

These plots diagnose periodic lag dependencies in the temperature series. Strong periodic spikes indicate seasonal patterns that models must capture.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

plot_acf(series, lags=60, ax=ax1, alpha=0.05,
         title='Autocorrelation Function (ACF) — Mean Temperature')
plot_pacf(series, lags=60, ax=ax2, alpha=0.05,
          title='Partial Autocorrelation Function (PACF) — Mean Temperature',
          method='ywm')

fig.tight_layout()
plt.show()

**ACF interpretation:** The slow, sinusoidal decay confirms strong positive autocorrelation at short lags (consecutive days are similar) and a seasonal pattern. This is characteristic of a non-stationary seasonal process.

**PACF interpretation:** Significant spikes at the first few lags suggest an AR component is appropriate. The pattern guides the choice of `p` in ARIMA/SARIMA models.

### Augmented Dickey-Fuller (ADF) Stationarity Test

In [ ]:
# ADF test for both temperature and precipitation
for col_name, col_label in [('temp_mean_c', 'Mean Temperature'),
                              ('precipitation_mm', 'Precipitation')]:
    series = df[col_name].dropna()
    result = adfuller(series, autolag='AIC')
    
    print(f'\n{"="*50}')
    print(f'ADF Test — {col_label}')
    print(f'{"="*50}')
    print(f'  Test Statistic : {result[0]:.4f}')
    print(f'  p-value        : {result[1]:.6f}')
    print(f'  Lags Used      : {result[2]}')
    print(f'  Observations   : {result[3]}')
    for key, val in result[4].items():
        print(f'  Critical ({key:>3s}) : {val:.4f}')
    
    if result[1] < 0.05:
        print(f'  → ✅ STATIONARY (p < 0.05)')
    else:
        print(f'  → ⚠️ NON-STATIONARY (p ≥ 0.05) — differencing recommended')

**Stationarity implications for modeling:**

- **Temperature is non-stationary** (p > 0.05) due to its strong seasonal cycle. For ARIMA-family models, seasonal differencing (D=1, s=365) or explicit seasonal decomposition is needed. SARIMA, Prophet, or ML models with seasonal features are recommended.
- **Precipitation is stationary** (p ≈ 0.0) because it's predominantly zero with sporadic spikes — no systematic trend or unit root.

---

## 3.4 Relationships & Anomalies

### Correlation Analysis

In [ ]:
# Pearson correlation heatmap
corr = df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Feature Correlation Matrix — Cairo Weather', fontweight='bold')
fig.tight_layout()
plt.show()

print('\nStrongest correlations (excluding self):')
corr_unstacked = corr.where(~np.eye(len(corr), dtype=bool)).unstack().dropna()
corr_sorted = corr_unstacked.abs().sort_values(ascending=False)
seen = set()
for (c1, c2), val in corr_sorted.items():
    pair = tuple(sorted([c1, c2]))
    if pair not in seen:
        seen.add(pair)
        actual = corr.loc[c1, c2]
        print(f'  {c1:25s} ↔ {c2:25s}: {actual:+.3f}')
    if len(seen) >= 5:
        break

### Rolling Z-Score Outlier Detection

Traditional Z-scores using the global mean would flag all summer days as "outliers" and all winter days as normal — ignoring seasonal structure. Instead, we use a **30-day rolling window** to compute local Z-scores, detecting values that are unusual *relative to their seasonal context*.

In [ ]:
# Rolling Z-score outlier detection
col = 'temp_mean_c'
window = 30
threshold = 3.0

rolling_mean = df[col].rolling(window, center=True).mean()
rolling_std_vals = df[col].rolling(window, center=True).std().replace(0, np.nan)
z_scores = (df[col] - rolling_mean) / rolling_std_vals
outlier_mask = z_scores.abs() > threshold

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(df.index, z_scores, color='#7f8c8d', linewidth=0.5, alpha=0.7, label='Rolling Z-score')
ax.axhline(y=threshold, color='#e74c3c', linestyle='--', alpha=0.7, label=f'±{threshold}')
ax.axhline(y=-threshold, color='#e74c3c', linestyle='--', alpha=0.7)

if outlier_mask.any():
    ax.scatter(df.index[outlier_mask], z_scores[outlier_mask], color='#e74c3c',
               s=40, zorder=5, label=f'Outliers ({outlier_mask.sum()})')

ax.set_ylabel('Z-score')
ax.set_title(f'Rolling Z-score Outlier Detection — Mean Temperature (window={window}d)', fontweight='bold')
ax.legend()
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

# Print outliers
if outlier_mask.any():
    print(f'\nOutliers detected ({outlier_mask.sum()}):')
    outlier_df = df[outlier_mask][['temp_mean_c']].copy()
    outlier_df['z_score'] = z_scores[outlier_mask]
    print(outlier_df.to_string())
else:
    print('\nNo outliers detected.')

### Sanity Checks

Verifying data integrity against physical constraints to rule out sensor/API artifacts.

In [ ]:
# Sanity checks
print('Data Sanity Checks')
print('=' * 50)

checks = {
    'Negative precipitation': (df['precipitation_mm'] < 0).sum(),
    'Temperature inversion (min > max)': (df['temp_min_c'] > df['temp_max_c']).sum(),
    'Humidity outside [0, 100]': ((df['humidity_pct'] < 0) | (df['humidity_pct'] > 100)).sum(),
    'Cloud cover outside [0, 100]': ((df['cloud_cover_pct'] < 0) | (df['cloud_cover_pct'] > 100)).sum(),
    'Temp > 55°C': (df['temp_max_c'] > 55).sum(),
    'Temp < -10°C': (df['temp_min_c'] < -10).sum(),
}

for check, count in checks.items():
    status = '✅' if count == 0 else f'⚠️ {count} violations'
    print(f'  {check:40s}: {status}')

print(f'\nOverall: {"ALL CHECKS PASSED" if all(v == 0 for v in checks.values()) else "ISSUES FOUND"}')

---

# Summary & Next Steps

## Key Findings

1. **Strong annual seasonality** in temperature with a clean sinusoidal pattern — ideal for SARIMA or seasonal ML features
2. **Temperature is non-stationary** (ADF p=0.28) — seasonal differencing or detrending required for ARIMA models
3. **Precipitation is stationary but zero-inflated** (94% dry days) — requires classification (rain/no-rain) + regression approach
4. **High feature correlation** — temperature variables are highly correlated; feature selection/PCA may be beneficial
5. **Data quality is excellent** — no gaps, no artifacts, 1 mild outlier in 3 years

## Next Steps (Future Phases)

- **Feature engineering:** Lag features, rolling statistics, seasonal indicators, Fourier terms
- **Baseline models:** Seasonal naïve, SARIMA, Prophet
- **ML models:** XGBoost, LSTM, Transformer-based architectures
- **Evaluation:** MAE, RMSE, MAPE with walk-forward cross-validation
- **Deployment:** API endpoint for real-time forecasts

---

*Data source: Open-Meteo Historical Weather API (ECMWF ERA5 reanalysis). Licensed under CC BY 4.0.*